# SALM Fine-tuning 00 · Overview and setup

This series adapts one speech LLM to sixteen new tasks, one notebook per task. A speech LLM trained for
transcription already hears well, but it does not know your language, your label set, your output format or your
annotators' conventions. Each notebook closes that gap with a short fine-tune on a public benchmark and measures the
result fairly.

This notebook covers the model, what you need, the pipeline every tutorial follows, and the silent failures to check
for. It ends with a quick hands-on look at the model.

## The model in one picture

```
   audio ──► [ speech encoder ] ──► [ projection ] ──► [ LLM backbone ] ──► text
              (often trained)        (always trained)    (frozen, adapted with LoRA r=32)
```

It's a **SALM** (`nemo.collections.speechlm2.models.SALMAutomodel`). The speech encoder turns audio into frames, the
projection maps those frames into the LLM's embedding space, and the LLM backbone writes the answer. The *prompt*
decides what that answer is: a transcript, a translation, an emotion, "unanswerable"…

The checkpoint used for the expected results is a strong English transcriber (1.5% WER on LibriSpeech test-clean) with
a fair amount of general audio knowledge.

## The sixteen tasks

| # | Task | Benchmark | Base → fine-tuned (test) |
|---|---|---|---|
| 01 | [New language: Swahili ASR](01_New_Language_ASR_Swahili.ipynb) | FLEURS `sw_ke` | 81.16 → 18.01 WER |
| 02 | [Speech translation En→Ca](02_Speech_Translation_English_Catalan.ipynb) | CoVoST 2 | 15.21 → 27.84 BLEU |
| 03 | [Spoken question answering](03_Spoken_Question_Answering.ipynb) | HeySQuAD | 43.30 → 84.37 F1 |
| 04 | [Intent classification](04_Intent_Classification_SLURP.ipynb) | SLURP | 50.57 → 87.91% |
| 05 | [Speech emotion](05_Speech_Emotion_CREMA-D.ipynb) | CREMA-D | 36.71 → 76.67% |
| 06 | [Keyword spotting](06_Keyword_Spotting_Speech_Commands.ipynb) | Speech Commands v2 | 94.78 → 97.42% |
| 07 | [Environmental sound](07_Environmental_Sound_ESC-50.ipynb) | ESC-50 | 40.25 → 62.00% |
| 08 | [Non-speech vocal sounds](08_Vocal_Sounds_VocalSound.ipynb) | VocalSound | 76.72 → 91.31% |
| 09 | [Music genre, by probing](09_Music_Genre_GTZAN_Probe.ipynb) | GTZAN | 78.97 → 86.21% |
| 10 | [Instrument family](10_Instrument_Family_NSynth.ipynb) | NSynth | 47.97 → 57.42% |
| 11 | [Spoken language ID](11_Language_ID_CommonLanguage.ipynb) | CommonLanguage | 49.81 → 92.87% |
| 12 | [Code-switching ASR](12_Code_Switching_ASR_ASCEND.ipynb) | ASCEND | 18.14 → 10.97 MER |
| 13 | [Accented conversational ASR](13_Accented_Conversational_ASR_EdAcc.ipynb) | EdAcc | 18.37 → 14.96 WER |
| 14 | [Pronunciation assessment](14_Pronunciation_Assessment_speechocean762.ipynb) | speechocean762 | r 0.217 → 0.678 |
| 15 | [Anti-spoofing](15_Anti-Spoofing_ASVspoof2019.ipynb) | ASVspoof 2019 LA | 30.73 → 0.57% EER |
| 16 | [Stuttering events](16_Stuttering_Events_SEP-28k.ipynb) | SEP-28k | 13.01 → 62.02 macro-F1 |

* **The biggest gains come where the base model can barely do the task.** A new language, translation, spoken QA,
  intents, emotions, language ID, pronunciation and stuttering improve by 40-380% relative.
* **Non-speech audio starts higher.** Sounds, vocalizations, music and instruments (07-10) start at 40-79% and gain
  9-54%.
* **Two exceptions.** On GTZAN (09) the base model is already near the state of the art, so the best move is a linear
  probe on the frozen model. On EdAcc (13) the base model is already good and most remaining errors are annotation
  conventions; part of the gain comes from the prompt (against the test-oracle prompt it is −8%).

**How the baseline is measured.** Each notebook scores the base model with two or three prompts. The baseline is the
test score of the prompt **chosen on dev**, the same way the fine-tuned checkpoint is chosen. Each notebook also prints
the **test-oracle**: the score of the best prompt on test, an upper bound for prompting alone. Every decision is made
on dev, and test is decoded once per candidate (each baseline prompt and the selected model).

## What you need

| | |
|---|---|
| GPU | one large GPU. The expected results were measured on a single NVIDIA GB300. Recipes use 70-160 GB of GPU memory during training |
| Training env | NeMo Speech with `speechlm2` (this repo), plus `cut-cross-entropy` for the fused CE loss |
| Serving env | **vLLM 0.28.0** with this repo installed (0.29 is not supported by the SpeechLM vLLM plugin) |
| Checkpoint | a SALM checkpoint as a directory in HuggingFace format, exported by `SALMAutomodel` |
| Disk | ~400 GB free per tutorial while it runs (~66 GB per training checkpoint, ~63 GB per exported model). The tools delete training checkpoints once exported, and each notebook ends with `tidy()`, which keeps only the selected model and deletes the other exports |

Install vLLM in a **separate** Python environment. vLLM pins its own `torch` version, which usually conflicts with the
one NeMo training uses. The notebooks call each environment through its own Python interpreter. The
[fine-tuning guide](../../../examples/speechlm2/finetuning/docs/fine-tuning-guide.md#2-environment) has the install
commands.

Paths are read from environment variables:

* `SALM_CHECKPOINT` (**required**, no default): the base SALM checkpoint directory (HuggingFace format)
* `SALM_FT_WORK` (default `tutorials/speechlm2/finetuning/workdir`): data, manifests, experiments, exports
* `TRAIN_PYTHON` (default `python`): the Python interpreter of the NeMo training environment
* `VLLM_PYTHON` (default `python`): the Python interpreter of the vLLM environment, e.g. `/path/to/vllm-env/bin/python`
* `VLLM_GPU_UTIL` (default 0.85): fraction of GPU memory vLLM may use; lower it (e.g. 0.40) to decode while something
  else uses the GPU

For example:

```bash
export SALM_CHECKPOINT=/path/to/salm-checkpoint
export VLLM_PYTHON=/path/to/vllm-env/bin/python
jupyter lab
```

In [ ]:
import json, os, shlex, subprocess, sys, time
from pathlib import Path


def find_nemo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "examples" / "speechlm2" / "finetuning").is_dir():
            return candidate
    raise RuntimeError(f"Could not locate the NeMo source tree from {start}. Open this notebook from inside the repo checkout.")


NEMO_ROOT = find_nemo_root(Path.cwd())
FT = NEMO_ROOT / "examples" / "speechlm2" / "finetuning"              # the tools used below
WORK = Path(os.environ.get("SALM_FT_WORK", NEMO_ROOT / "tutorials" / "speechlm2" / "finetuning" / "workdir")).absolute()
os.environ["SALM_FT_WORK"] = str(WORK)                                # data, manifests, experiments, exports
os.environ.pop("MPLBACKEND", None)  # the kernel's inline plotting backend would break matplotlib in child processes
if not os.environ.get("SALM_CHECKPOINT"):
    raise RuntimeError(
        "SALM_CHECKPOINT is not set. Point it at your SALM checkpoint: a directory in HuggingFace format "
        "exported by SALMAutomodel. Run `export SALM_CHECKPOINT=/path/to/checkpoint` before "
        "starting Jupyter, or set os.environ['SALM_CHECKPOINT'] in a cell above this one."
    )
CKPT = Path(os.environ["SALM_CHECKPOINT"]).expanduser().absolute()  # base SALM checkpoint (HuggingFace format)
if not CKPT.is_dir():
    raise RuntimeError(f"SALM_CHECKPOINT={CKPT} is not a directory. It must be a SALM checkpoint in HuggingFace format.")
PYTHON = os.environ.get("TRAIN_PYTHON", "python")                     # NeMo training environment
VLLM_PYTHON = os.environ.get("VLLM_PYTHON", "python")                 # vLLM 0.28 environment (ideally a separate one)
M, EXP, EXPORT, EVAL = (WORK / d for d in ("manifests", "experiments", "exports", "eval"))
q = shlex.quote                                                        # shell-quote prompts (keeps non-ASCII intact)
for d in (M, EXP, EXPORT, EVAL, WORK / "logs", WORK / "configs"):
    d.mkdir(parents=True, exist_ok=True)


RETRIES = int(os.environ.get("SALM_FT_RETRIES", 3))                   # raise when other jobs share the GPU


def sh(cmd, log=None, retries=1):
    """Run a shell command from the NeMo root. With `log`, output goes to a file and only its tail is shown."""
    for attempt in range(1, retries + 1):
        print(f"$ {cmd}" + (f"   > {log}" if log else ""), flush=True)
        if log:
            with open(WORK / "logs" / log, "w") as f:
                rc = subprocess.run(cmd, shell=True, cwd=NEMO_ROOT, stdout=f, stderr=subprocess.STDOUT).returncode
            tail = (WORK / "logs" / log).read_text(errors="ignore").splitlines()
            print("\n".join(l for l in tail[-400:] if l.startswith(("[eval]", "[export]", "[scores]", "[merge]", "wrote")))[-3000:])
        else:
            rc = subprocess.run(cmd, shell=True, cwd=NEMO_ROOT).returncode
        if rc == 0:
            return
        if attempt < retries:  # e.g. a vLLM engine starting while another job allocates GPU memory
            print(f"(attempt {attempt} failed, retrying in 90 s)")
            time.sleep(90)
    raise RuntimeError(f"command failed (rc={rc}): {cmd}" + (f"; see {WORK / 'logs' / log}" if log else ""))


def evaluate(model, manifest, name, task_args, extra=""):
    """Decode `manifest` with vLLM (greedy) and return the metric summary.

    Re-running a cell reuses a previous result only if its fingerprint (model, manifest, prompt, decoder and scoring
    settings, code revision) matches; a result produced from different inputs is an error, never silently reused.
    """
    out = EVAL / f"{name}.jsonl"
    sh(f"{VLLM_PYTHON} {FT}/vllm_task_eval.py --model {model} --manifest {manifest} --out {out} {task_args} {extra} "
       f"--reuse", log=f"{name}.log", retries=RETRIES)
    return json.load(open(f"{out}.summary.json"))


def tidy(keep, name):
    """Free disk: delete this tutorial's other merged exports (~63 GB each) and leftover training checkpoints."""
    import shutil

    for p in EXPORT.glob(f"{name}_*"):
        if p != keep:
            shutil.rmtree(p)
    shutil.rmtree(EXP / name / "checkpoints", ignore_errors=True)
    print(f"kept {keep}; removed the other {name} exports and checkpoints")


def samples(name, n=3, keys=("text", "pred_text")):
    """Print a few decoded examples."""
    rows = [json.loads(l) for l in open(EVAL / f"{name}.jsonl")]
    for r in rows[:n]:
        print("  |  ".join(f"{k}: {str(r.get(k))[:110]}" for k in keys))


def result(name, value, expected, tol):
    """Compare a number with the expected result, and record it in results.json."""
    ok = abs(value - expected) <= tol
    print(f"{'✅' if ok else '❌'} {name}: {value:.4f}  (expected: {expected} ± {tol})")
    res = WORK / "results.json"
    d = json.loads(res.read_text()) if res.exists() else {}
    d[name] = {"value": value, "expected": expected, "tolerance": tol, "ok": ok}
    res.write_text(json.dumps(d, indent=1))
    return ok


print("NeMo root:", NEMO_ROOT, "\nwork dir: ", WORK, "\ncheckpoint:", CKPT)

## Environment checks

CUDA's device order can differ from `nvidia-smi`'s, for example when a display GPU sits next to the compute GPU. Make
sure device 0 is the big one. The `Conv1d` check catches a cuDNN wheel/system mismatch that would break every
convolution in the model.

In [ ]:
sh(f'{PYTHON} -c "import torch; print(torch.cuda.get_device_properties(0).name)"')
sh(f'{PYTHON} -c "import torch; torch.nn.Conv1d(192,1536,3,padding=1).cuda()(torch.randn(2,192,100,device=\'cuda\')); print(\'cuDNN OK\')"')
sh(f'{VLLM_PYTHON} -c "import vllm; print(\'vLLM\', vllm.__version__)"')
print("checkpoint files:", sorted(p.name for p in CKPT.iterdir()))

## The pipeline every tutorial follows

```
 1. data      HF hub ─ prepare_benchmarks.py ─► 16 kHz FLAC + NeMo manifest (*.json) with a `context` prompt per row
 2. baseline  base model × 2-3 prompts × {dev, test} ─► baseline = test score of the best dev prompt
 3. config    make_ft_config.py --prompt manifest   (derived from the checkpoint, never from an example YAML)
 4. train     run_training.py --preflight --smoke   (coverage check ─► 20-step smoke ─► full run with a watchdog)
 5. select    export_checkpoint.py (merge LoRA) ─► vllm_task_eval.py on dev for each candidate ─► pick by the decoded metric
 6. test      decode test once, plus the LibriSpeech guard
```

Every notebook checks its numbers against the expected results (`✅ / ❌`). They match within a tolerance, not bit
for bit:

* vLLM batches requests dynamically, and a different batch composition changes floating-point rounding enough to flip
  ~1% of greedy decodes.
* Training is not bit-for-bit reproducible, so fine-tuned results vary from run to run.

Baselines are checked to ±0.5 points (up to ±1.5 on the smallest test sets), fine-tuned results to a per-task margin.

All tools live in [`examples/speechlm2/finetuning/`](../../../examples/speechlm2/finetuning/).

## Seven ways a fine-tune fails silently, and the check for each

| What you'd see | What's really going on | The check |
|---|---|---|
| Fine-tune scores exactly like the base model | LoRA never applied at serving time | `export_checkpoint.py` merges it into the weights and asserts `lora_B ≠ 0` |
| Plausible but awful output | encoder weights never loaded (config ≠ checkpoint) | `run_training.py --preflight` |
| No `val_loss`, "not in top k" forever | validation never ran (`limit_train_batches` ≥ batches/epoch) | `epoch_batches.py` + the watchdog in `run_training.py` |
| The model answers one class for everything | label-sorted data (shards or manifest) | shuffle on disk (`shuffle_manifest.py`); tutorials 06, 11 |
| Worse than base, but only on a few clips | greedy repetition loops | `--max-tokens-per-sec` (tutorial 13) |
| The baseline looks terrible | the scorer rejects correct answers ("angry" vs "anger") | read the confusion matrix and raw outputs (tutorial 05) |
| A big gain from a handful of rows | references that are scoring markers (`IGNORE_TIME_SEGMENT_IN_SCORING`), learned verbatim | grep the references; read the rows with the largest errors (tutorial 13) |

## Hands-on: meet the model

Decode three LibriSpeech clips with the base model: once as plain transcription, then with two other prompts. The
prompt decides the task. The manifests carry it in their `context` field.

In [ ]:
sh(f"{PYTHON} {FT}/prepare_benchmarks.py librispeech", log="prep_ls.log")
rows = [json.loads(l) for l in open(M / "librispeech" / "test_clean.json")][:3]
demo = M / "librispeech" / "demo3.json"
demo.write_text("".join(json.dumps(r) + "\n" for r in rows))
for name, prompt in [("transcribe", "Provide a verbatim transcript of the audio."),
                     ("french", "Translate the English audio into French."),
                     ("summary", "Summarize what the speaker says in five words.")]:
    evaluate(CKPT, demo, f"demo_{name}", "--task asr --max-tokens 64", f"--prompt {q(prompt)}")
    print(f"\n--- {prompt}")
    samples(f"demo_{name}", n=3, keys=("pred_text",))
print("\n--- reference")
for r in rows:
    print(r["text"])

## Takeaways

* The prompt decides the task. Every tutorial starts by finding the best prompt for the base model.
* Every tutorial follows the same pipeline: data → fair baseline → config from the checkpoint → train with checks →
  select on dev by decoding → test once.
* Most fine-tuning failures are silent. Run the checks in the table above.

Pick any tutorial next. They're independent: each downloads its own data, measures its own baseline and checks its
own numbers.